# C++ 第 3 章：类与对象

> **开始前：选择 C++17 内核。** 本章不能使用 C17 内核。在线实验使用固定数据或 `std::istringstream`，键盘输入 `std::cin` / `std::getline(std::cin, ...)` 放在本地完整程序中练习。

**先修**：C 部分的变量、控制语句、数组、指针和函数；按 C++ 章节顺序学习。

**本章目标**：区分类与对象；用构造函数维护有效状态；理解 const 成员及运算符。

**学习方法**：先预测，再运行、修改并解释。块内局部变量可重复运行；修改函数、类或模板后，使用 Kernel → Restart Kernel and Run All Cells。各章互不依赖内核状态。

**保存作业**：见 [课程入口](https://github.com/luckymark/c_course#保存作业与继续学习)。

## 1. 从公开数据到带约束的类型

C++ 的 struct 和 class 都能有成员函数，默认访问权限分别为 public 和 private。公开的数据记录可用 struct；需要保证约束的对象用私有数据与构造函数。对象拥有自己的状态，成员函数中的 this 指向当前对象。

以下 Grade 保证分数始终在0..100。初始化列表直接初始化成员；构造函数没有返回类型。`explicit` 阻止把整数隐式转换为 Grade，调用处写清 `Grade{85}`。const 成员函数承诺不修改普通成员，可用于 const 对象。

In [ ]:
#include <stdexcept>
class Grade {
public:
    explicit Grade(int value) : value_{value} {
        if (value < 0 || value > 100) { throw std::out_of_range("score must be 0..100"); }
    }
    int value() const { return value_; }
    bool passed() const { return value_ >= 60; }
    bool operator==(const Grade& other) const { return value_ == other.value_; }
private:
    int value_;
};

In [ ]:
#include <iostream>
{
    const Grade first{85};
    const Grade copy = first;
    std::cout << first.value() << ' ' << copy.passed() << ' ' << (first == copy) << '\n';
}

## 2. 成员组合与生命周期

Student 包含姓名与 Grade，这是“有一个”的组合关系。局部对象离开作用域时自动析构；string 等成员也随之清理，通常不需要手写析构、拷贝和赋值（Rule of Zero）。`static` 成员属于类型，不是每个对象单独一份；本例不用共享可变状态。

运算符重载应保留直观含义，本例 == 比较分数值。不要为了练习重载而赋予 +、== 意外含义。

In [ ]:
#include <string>
struct Student {
    std::string name;
    Grade grade;
};

In [ ]:
#include <iostream>
{
    const Student student{"Alice", Grade{60}};
    std::cout << student.name << ' ' << student.grade.passed() << '\n';
    try { const Grade invalid{-1}; }
    catch (const std::out_of_range& error) { std::cout << error.what() << '\n'; }
}

## 小练习：给一个局部类型补上只读查询

只填写 BEGIN ANSWER 与 END ANSWER 之间的代码。先预测，再运行；初始 FAIL 表示尚未作答。完成后应全部 PASS。

In [ ]:
#include <iostream>
{
    class Result {
    public:
        explicit Result(int score) : score_{score} {}
        bool passed() const {
            /* BEGIN ANSWER */
            (void)score_; // 占位：完成答案时替换本行与下一行。
            return false;
            /* END ANSWER */
        }
    private:
        int score_;
    };
    for (const int score : {60, 100}) {
        const Result result{score};
        std::cout << score << ": " << (result.passed() ? "PASS" : "FAIL") << '\n';
    }
    std::cout << "59: " << (!Result{59}.passed() ? "PASS" : "FAIL") << '\n';
}

<details><summary>提示</summary>

先检查空数据和边界，再处理一般情况；只计算题目需要的返回值，不改变输入。

</details>

<details><summary>参考答案：完成后再展开</summary>

```cpp
return score_ >= 60;
```

</details>

## 本地实践：完整 C++17 程序

把代码保存为 `lesson.cpp`，在终端执行 `c++ -std=c++17 -Wall -Wextra -Wpedantic lesson.cpp -o lesson`，编译成功后运行 `./lesson`。Windows GCC 使用 `g++` 和 `-o lesson.exe`，PowerShell 运行 `.\lesson.exe`。完整程序不要直接放入在线单元或手动调用 main。

```cpp
#include <iostream>
#include <stdexcept>
class Grade {
public:
    explicit Grade(int value) : value_{value} {
        if (value < 0 || value > 100) { throw std::out_of_range("invalid"); }
    }
    bool passed() const { return value_ >= 60; }
private:
    int value_;
};
int main() {
    int value{};
    if (!(std::cin >> value)) { std::cout << "invalid\n"; return 1; }
    try {
        const Grade grade{value};
        std::cout << (grade.passed() ? "pass" : "fail") << '\n';
    } catch (const std::out_of_range& error) {
        std::cout << error.what() << '\n'; return 1;
    }
}
```

验收输入与输出（`\n` 表示换行）：

- 输入 `"60"` → 输出 `"pass\n"`，退出码 0。
- 输入 `"59"` → 输出 `"fail\n"`，退出码 0。
- 输入 `"-1"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"101"` → 输出 `"invalid\n"`，退出码 1。

## 分层练习与自查

1. **基础**：为什么 const Grade 能调用 passed()，却不能随意调用非 const 成员函数？
2. **改错**：在独立副本中尝试读取 private 成员，解释诊断。
3. **应用**：增加返回等级字符串的 const 成员，覆盖59/60/89/90/100。
4. **自查**：构造失败后是否会得到一个可用的无效 Grade？为什么不能把分数字段改成 public 随意赋值？